In [1]:
import json
import math
import os
import re
import string
from google import genai

OUTPUT_PATH = "./results/small_test.jsonl"

# load dataset
public_data = [json.loads(line) for line in open("./data/public.jsonl")]

n_mcq  = sum(bool(d.get("options")) for d in public_data)
n_free = sum(not d.get("options")   for d in public_data)
print(f"Loaded {len(public_data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

Loaded 1126 questions  (375 MCQ, 751 free-form)


In [9]:
# prompts for free response and MCQ problems
SYSTEM_PROMPT_FRQ = (
    "Be mathematically correct. Be brief but complete. "
    "Show the visible reasoning. "
    "End with exactly one final answer in \\boxed{}. "
    "If there are multiple answers, put them in one \\boxed{} separated by commas. "
    "The final boxed answer must exactly match the known correct answer. "
    "Do not mention the known answer explicitly. "
    "Preserve 1e-8 precision if needed."
)

SYSTEM_PROMPT_MCQ = (
    "Be mathematically correct. Be brief but complete. "
    "Show the visible reasoning. "
    "Use the answer choices to determine the correct option. "
    "End with exactly one final answer in the form \\boxed{<letter>}. "
    "The final boxed answer must exactly match the known correct answer. "
    "Do not mention the known answer explicitly. "
)

FRQ_TEMPLATE = """Question: {question}, Known correct option: {answer}"""
MCQ_TEMPLATE = """Question: {question}, Options: {options}, Known correct option: {answer}"""

from typing import Optional

def build_prompt(row):
    question = row["question"]
    answer   = row.get("answer")
    options  = row.get("options")
    print(f"Options: {options}\n")

    return build_prompt_internal(question, options, answer)

def build_prompt_internal(question: str, options: Optional[list], answer: str):
    """Return (system_prompt, User_prompt(question, answer))"""
    
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return (SYSTEM_PROMPT_MCQ, MCQ_TEMPLATE.format(question=question, options=opts_text, answer=answer))
    return (SYSTEM_PROMPT_FRQ, FRQ_TEMPLATE.format(question=question, answer=answer))

In [10]:
results = []

for row in public_data[:10]:
    system_prompt, user_prompt = build_prompt(row)

    example = {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
            {"role": "assistant", "content": "a"},
        ]
    }

    results.append(example)
    print(f"done id={row['id']}")

Options: None

done id=0
Options: ['$0$', '$frac{1}{a}$', '$frac{3}{a}$', '$frac{1}{2a^2}$', '$frac{1}{2a}$', '$frac{2}{a}$', '$2a$', '$frac{3}{2a}$', '$frac{3}{2a^2}$', '$frac{1}{a^2}$']

done id=1
Options: None

done id=2
Options: None

done id=3
Options: ['$$\n( 6+4 \\mathrm{i} ) z^{5}\n$$', '$$\n( 4-5 \\mathrm{i} ) z^{6}\n$$', '$$\n( 1-2 \\mathrm{i} ) z^{3}\n$$', '$$\n( 2+ \\mathrm{i} ) z^{3}\n$$', '$$\n( 5+3 \\mathrm{i} ) z^{7}\n$$', '$$\n( 1+2 \\mathrm{i} ) z^{2}\n$$', '$$\n( 2- \\mathrm{i} ) z^{8}\n$$', '$$\n( 3-2 \\mathrm{i} ) z^{2}\n$$', '$$\n( 2-3 \\mathrm{i} ) z^{4}\n$$', '$$\n( 3+4 \\mathrm{i} ) z^{5}\n$$']

done id=4
Options: None

done id=5
Options: None

done id=6
Options: None

done id=7
Options: None

done id=8
Options: ['$6$ hours', '$9$ hours', '$24$ hours', '$15$ hours', '$3$ hours', '$1$ hours', '$12$ hours', '$18$ hours', '$2$ hours', '$21$ hours']

done id=9


In [5]:
# from google import genai
# import os

# client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

# def generate_response(system_prompt, user_prompt):
#     response = client.models.generate_content(
#         model="gemini-2.5-flash",
#         contents=[
#             {"role": "user", "parts": [{"text": system_prompt + "\n\n" + user_prompt}]}
#         ],
#     )
#     return response.text.strip()

In [6]:
# results = []

# for row in public_data[:10]:   # ← start with 10
#     system_prompt, user_prompt = build_prompt(row)

#     assistant = generate_response(system_prompt, user_prompt)

#     example = {
#         "messages": [
#             {"role": "system", "content": system_prompt},
#             {"role": "user", "content": row["question"]},
#             {"role": "assistant", "content": assistant},
#         ]
#     }

#     results.append(example)
#     print(f"done id={row['id']}")